# Seq2Seq Ubuntu Chatbot — Dual-Model Comparison

Loads baseline and attention models simultaneously and shows both responses side by side.
Supports greedy, top-p, and beam search decoding.

In [ ]:
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn.functional as F
import sentencepiece as spm

from config import CONFIG
from models import build_model
from phase1 import _clean_text

In [ ]:
@torch.inference_mode()
def _greedy_decode(model, src, src_lengths, sos_idx, eos_idx, max_len, device):
    from evaluate import greedy_decode
    return greedy_decode(model, src, src_lengths, sos_idx, eos_idx, max_len, device)


@torch.inference_mode()
def _topp_decode(model, src, src_lengths, sos_idx, eos_idx, max_len, device,
                 top_p, temperature, ngram_block):
    from evaluate import top_p_decode
    return top_p_decode(model, src, src_lengths, sos_idx, eos_idx, max_len,
                        device, top_p, temperature, ngram_block)


@torch.inference_mode()
def beam_decode(
    model,
    src: torch.Tensor,
    src_lengths: torch.Tensor,
    sos_idx: int,
    eos_idx: int,
    max_len: int,
    device: torch.device,
    beam_width: int = 5,
    length_penalty: float = 0.7,
) -> List[List[int]]:
    """Beam search decoding for a single input (batch_size=1)."""
    model.eval()
    src = src.to(device)
    src_lengths = src_lengths.to(device)

    enc_out, (h_n, c_n) = model.encoder(src, src_lengths)
    src_mask = (src == model.encoder.embedding.padding_idx)
    dec_h, dec_c = model.bridge(h_n, c_n)
    enc_dim = enc_out.size(-1)
    context = torch.zeros(1, enc_dim, device=device)

    sos_tok = torch.tensor([sos_idx], dtype=torch.long, device=device)
    logits, h1, c1, ctx1, _ = model.decoder.forward_step(
        sos_tok, dec_h, dec_c, enc_out, context, src_mask
    )
    log_probs = F.log_softmax(logits[0], dim=-1)
    topk_lp, topk_ids = log_probs.topk(min(beam_width, log_probs.size(-1)))

    active: List[dict] = []
    completed: List[dict] = []

    for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
        if tid == eos_idx:
            completed.append({"score": lp, "tokens": []})
        else:
            active.append({"score": lp, "tokens": [tid],
                           "h": h1, "c": c1, "ctx": ctx1})

    for _ in range(max_len - 1):
        if not active:
            break

        candidates: List[dict] = []
        for beam in active:
            inp = torch.tensor([beam["tokens"][-1]], dtype=torch.long, device=device)
            logits, new_h, new_c, new_ctx, _ = model.decoder.forward_step(
                inp, beam["h"], beam["c"], enc_out, beam["ctx"], src_mask
            )
            lp_all = F.log_softmax(logits[0], dim=-1)
            topk_lp, topk_ids = lp_all.topk(min(beam_width, lp_all.size(-1)))

            for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
                new_score = beam["score"] + lp
                new_tokens = beam["tokens"] + [tid]
                if tid == eos_idx:
                    norm = len(new_tokens) ** length_penalty
                    completed.append({"score": new_score / max(norm, 1e-6),
                                      "tokens": beam["tokens"]})
                else:
                    candidates.append({"score": new_score, "tokens": new_tokens,
                                       "h": new_h, "c": new_c, "ctx": new_ctx})

        candidates.sort(
            key=lambda b: b["score"] / max(len(b["tokens"]) ** length_penalty, 1e-6),
            reverse=True,
        )
        active = candidates[:beam_width]

    for b in active:
        norm = len(b["tokens"]) ** length_penalty
        completed.append({"score": b["score"] / max(norm, 1e-6),
                          "tokens": b["tokens"]})

    if not completed:
        return [[]]
    completed.sort(key=lambda b: b["score"], reverse=True)
    return [completed[0]["tokens"]]

## Checkpoint Discovery & Model Loading

In [ ]:
def list_checkpoints(checkpoint_dir: Path, model_type: str) -> List[Path]:
    """Return sorted list of .pt checkpoints matching model_type."""
    found = []
    for pat in [f"{model_type}_best.pt", f"{model_type}_last.pt", f"{model_type}_step_*.pt"]:
        found.extend(checkpoint_dir.glob(pat))
    seen = set()
    ordered = []
    for priority in [f"{model_type}_best.pt", f"{model_type}_last.pt"]:
        p = checkpoint_dir / priority
        if p.exists() and p not in seen:
            ordered.append(p)
            seen.add(p)
    for p in sorted(checkpoint_dir.glob(f"{model_type}_step_*.pt")):
        if p not in seen:
            ordered.append(p)
            seen.add(p)
    return ordered


def load_model_from_checkpoint(
    ckpt_path: Path,
    model_type: str,
    config: dict,
    device: torch.device,
) -> Tuple[torch.nn.Module, dict]:
    model = build_model(model_type, config, device)
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt

## Context Building & Decode Dispatch

In [ ]:
def build_context(
    history: List[str],
    sp_processor,
    max_ctx_tokens: int,
    max_turns: int,
) -> torch.Tensor:
    """Encode conversation history as a [1, ctx_len] LongTensor."""
    turns = history[-max_turns:] if history else []
    cleaned = [_clean_text(t) for t in turns]
    cleaned = [t for t in cleaned if t]
    joined = " __eot__ ".join(cleaned) if cleaned else ""
    tokens = sp_processor.encode(joined, out_type=int) if joined else []
    tokens = tokens[-max_ctx_tokens:]
    if not tokens:
        tokens = [CONFIG.get("unk_idx", 1)]
    return torch.tensor([tokens], dtype=torch.long)


def decode(
    model,
    context_tensor: torch.Tensor,
    sp_processor,
    config: dict,
    device: torch.device,
    mode: str,
    beam_width: int,
) -> str:
    src = context_tensor.to(device)
    src_lengths = torch.tensor([src.size(1)], dtype=torch.long)

    sos = config.get("sos_idx", 2)
    eos = config.get("eos_idx", 3)
    max_len = config.get("max_decode_len", 40)

    if mode == "greedy":
        ids = _greedy_decode(model, src, src_lengths, sos, eos, max_len, device)
    elif mode == "beam":
        ids = beam_decode(model, src, src_lengths, sos, eos, max_len, device,
                          beam_width=beam_width)
    else:
        ids = _topp_decode(model, src, src_lengths, sos, eos, max_len, device,
                           top_p=config.get("top_p", 0.9),
                           temperature=config.get("temperature", 0.8),
                           ngram_block=config.get("ngram_block", 3))

    toks = ids[0] if ids else []
    return sp_processor.decode(toks) if toks else "…"

## Setup & Load Models

Configure paths and load both the baseline and attention models.

In [ ]:
# Configuration
ckpt_dir = Path(CONFIG.get("checkpoint_dir", "checkpoints"))
art_dir  = Path(CONFIG.get("artifact_dir", "artifacts"))
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

decoding_mode = "topp"   # "greedy", "topp", or "beam"
beam_width = 5

print(f"Device         : {device}")
print(f"Checkpoint dir : {ckpt_dir}")
print(f"Artifact dir   : {art_dir}")

In [ ]:
# Load SentencePiece processor
sp_path = art_dir / "stage5_spm.model"
assert sp_path.exists(), f"SPM model not found: {sp_path}. Run phase1.py first."
sp_processor = spm.SentencePieceProcessor(model_file=str(sp_path))
print(f"SPM loaded: {sp_path}")

# Discover and load models
models: Dict[str, torch.nn.Module] = {}
for model_type in ["baseline", "attention"]:
    ckpts = list_checkpoints(ckpt_dir, model_type)
    if not ckpts:
        print(f"  No {model_type} checkpoints found — skipping")
        continue
    ckpt_path = ckpts[0]  # use best checkpoint
    model, ckpt = load_model_from_checkpoint(ckpt_path, model_type, CONFIG, device)
    epoch = ckpt.get("epoch", "?")
    vl = ckpt.get("val_loss", float("nan"))
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Loaded {model_type:<10} | {ckpt_path.name}  "
          f"(epoch={epoch}, val_loss={vl:.4f}, params={n_params:,})")
    models[model_type] = model

print(f"\n{len(models)} model(s) loaded.")

## Chat

Type a message and see both models respond. Change `decoding_mode` above to switch between greedy, top-p, and beam search.

In [ ]:
# Single-turn chat: edit the message and re-run this cell
user_message = "How do I install ubuntu on my laptop?"

history = [user_message]
context = build_context(
    history, sp_processor,
    CONFIG["max_ctx_tokens"],
    CONFIG["max_ctx_turns"],
).to(device)

print(f"You: {user_message}\n")
for name, model in models.items():
    response = decode(model, context, sp_processor, CONFIG, device,
                      decoding_mode, beam_width)
    print(f"  [{name}]  {response}")

In [ ]:
# Multi-turn chat: run this cell repeatedly to have a conversation
# (keeps history between runs)
try:
    chat_history
except NameError:
    chat_history = []
    print("Chat started. Run this cell for each turn.\n")

user_input = input("You: ")
chat_history.append(user_input)

context = build_context(
    chat_history, sp_processor,
    CONFIG["max_ctx_tokens"],
    CONFIG["max_ctx_turns"],
).to(device)

print()
first_response = None
for name, model in models.items():
    response = decode(model, context, sp_processor, CONFIG, device,
                      decoding_mode, beam_width)
    print(f"  [{name}]  {response}")
    if first_response is None:
        first_response = response

chat_history.append(first_response or "…")
print(f"\n  (history: {len(chat_history)//2} turns)")